In [5]:
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.transforms.functional as TF
import tqdm
from ultralytics import YOLO
import plotly.express as px

# Hardware Device Setup
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

# Dataset and Hyperparameter Configurations
DATASET_DIR = "Plot Status"
JSON_ANNOTATION_PATH = os.path.join(DATASET_DIR, "annotations.json")
CHECKPOINT_PATH = "dense_lejepa_yolov8_cotton_50k_checkpoint.pth"
OUTPUT_DIR = "./classification_results_50_50"

IMAGE_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 50
NUM_WORKERS = 0
TEST_SPLIT_RATIO = 0.50  # 50% Training / 50% Testing

LEARNING_RATE = 0.017711616697652244
WEIGHT_DECAY = 1.009744159203988e-06
DROPOUT_RATE = 0.3
RANDOM_SEED = 48

LABEL_MAPPING = {"headland": 0, "between_plots": 1, "in_plot": 2}
NUM_CLASSES = len(LABEL_MAPPING)
INV_LABEL_MAPPING = {v: k for k, v in LABEL_MAPPING.items()}
CLASS_NAMES = list(LABEL_MAPPING.keys())


class DefoliatedPlotDataset(Dataset):
    """
    Custom PyTorch Dataset for parsing and loading image-annotation pairs.
    Includes explicit LeJEPA-matched image normalization to prevent domain shift.
    """
    def __init__(self, root_dir, json_path, label_map, image_size=128):
        self.root_dir = Path(root_dir)
        self.image_size = image_size
        self.label_map = label_map

        if not os.path.exists(json_path):
            raise FileNotFoundError(f"Annotation file not found: {json_path}")

        with open(json_path, "r") as f:
            self.annotations = json.load(f)

        self.samples = []
        valid_exts = (".jpg", ".jpeg", ".png", ".bmp")

        for img_path in self.root_dir.rglob("*"):
            if img_path.suffix.lower() in valid_exts and img_path.name in self.annotations:
                str_label = self.annotations[img_path.name]
                if str_label in self.label_map:
                    self.samples.append((img_path, self.label_map[str_label]))

        if len(self.samples) == 0:
            raise RuntimeError(f"No valid image-annotation pairs found in {self.root_dir}")

        print(f"Total Dataset loaded: {len(self.samples)} valid samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label_idx = self.samples[idx]

        orig_img = Image.open(img_path).convert("RGB")
        img_resized = orig_img.resize((self.image_size, self.image_size))

        # Scale to [0, 1] range
        img_tensor = torch.from_numpy(np.array(img_resized)).permute(2, 0, 1).float() / 255.0
        
        # APPLY LEJEPA NORMALIZATION (Critical for feature extraction alignment)
        img_tensor = TF.normalize(
            img_tensor, 
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225]
        )

        label_tensor = torch.tensor(label_idx, dtype=torch.long)

        return img_tensor, label_tensor, str(img_path)


class LinearPlotClassifier(nn.Module):
    """
    Linear Probe Classifier architecture leveraging a frozen pre-trained backbone.
    """
    def __init__(self, checkpoint_path=None, num_classes=NUM_CLASSES, dropout_rate=0.3):
        super().__init__()
        
        yolo = YOLO("yolov8n.pt").model
        self.backbone = nn.Sequential(*list(yolo.model[:10]))

        if checkpoint_path and os.path.exists(checkpoint_path):
            try:
                state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
            except Exception:
                import numpy._core.multiarray
                torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])
                state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

            # Strip the LeJEPA specific namespace to match standard PyTorch nn.Sequential numbering
            backbone_dict = {
                k.replace("backbone.layers.", ""): v
                for k, v in state_dict.items()
                if k.startswith("backbone.layers.")
            }
            
            self.backbone.load_state_dict(backbone_dict, strict=False)

        # Freeze the backbone completely for the linear probe
        for param in self.backbone.parameters():
            param.requires_grad = False

        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
            feat_channels = self.backbone(dummy).shape[1]

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(feat_channels, num_classes)

    def extract_features(self, x):
        feats = self.backbone(x)
        pooled = self.pool(feats).flatten(1)
        return pooled

    def forward(self, x):
        pooled = self.extract_features(x)
        dropped = self.dropout(pooled)
        logits = self.classifier(dropped)
        return logits


@torch.no_grad()
def evaluate_model(model, dataloader, device):
    """
    Evaluates the model on the provided dataloader.
    Returns predictions, true labels, and the average loss.
    """
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    for images, labels, _ in dataloader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        
        preds = torch.argmax(logits, dim=1)
        
        total_loss += loss.item()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    return all_preds, all_labels, avg_loss


def generate_classification_metrics(all_labels, all_preds, class_names, output_dir):
    """
    Generates and saves the classification report and confusion matrix.
    """
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4)

    print("\n" + "="*50)
    print(f"TEST SET CLASSIFICATION REPORT (50% SPLIT)")
    print("="*50)
    print(report)

    print("\n" + "="*50)
    print("TEST SET CONFUSION MATRIX")
    print("="*50)
    print(cm)

    os.makedirs(output_dir, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set_title("Confusion Matrix (50% Test Set)")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    plt.tight_layout()

    fig_path = os.path.join(output_dir, "test_confusion_matrix.png")
    plt.savefig(fig_path, dpi=300)
    plt.close()
    print(f"Confusion Matrix plot saved to: {fig_path}")


@torch.no_grad()
def plot_pca_embeddings_2d_and_3d(model, dataloader, device, class_names, output_dir):
    """
    Extracts latent embeddings for the Test dataset and constructs PCA plots.
    """
    model.eval()
    all_features = []
    all_labels = []

    for images, labels, _ in tqdm.tqdm(dataloader, desc="Extracting Test Embeddings for PCA"):
        images = images.to(device)
        feats = model.extract_features(images)
        all_features.append(feats.cpu().numpy())
        all_labels.append(labels.numpy())

    all_features = np.concatenate(all_features, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    label_names = [INV_LABEL_MAPPING[lbl] for lbl in all_labels]

    os.makedirs(output_dir, exist_ok=True)

    # -------------------------
    # 2D PCA Dimensionality Reduction
    # -------------------------
    pca_2d = PCA(n_components=2, random_state=RANDOM_SEED)
    embeddings_2d = pca_2d.fit_transform(all_features)
    var_ratio_2d = pca_2d.explained_variance_ratio_ * 100

    df_pca_2d = pd.DataFrame({
        "PCA Component 1": embeddings_2d[:, 0],
        "PCA Component 2": embeddings_2d[:, 1],
        "Plot Status": label_names,
    })

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(
        data=df_pca_2d, x="PCA Component 1", y="PCA Component 2",
        hue="Plot Status", style="Plot Status", palette="Set2", alpha=0.8, s=60, ax=ax,
    )

    ax.set_title(f"2D PCA Projection (Test Set)\nVariance: PC1={var_ratio_2d[0]:.2f}%, PC2={var_ratio_2d[1]:.2f}%", fontweight="bold")
    ax.set_xlabel(f"PCA Component 1 ({var_ratio_2d[0]:.2f}% Variance)")
    ax.set_ylabel(f"PCA Component 2 ({var_ratio_2d[1]:.2f}% Variance)")
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()

    pca_2d_path = os.path.join(output_dir, "pca_embeddings_2d_test.png")
    plt.savefig(pca_2d_path, dpi=300)
    plt.close()

    # -------------------------
    # 3D PCA Dimensionality Reduction
    # -------------------------
    pca_3d = PCA(n_components=3, random_state=RANDOM_SEED)
    embeddings_3d = pca_3d.fit_transform(all_features)
    var_ratio_3d = pca_3d.explained_variance_ratio_ * 100

    df_pca_3d = pd.DataFrame({
        "PC1": embeddings_3d[:, 0],
        "PC2": embeddings_3d[:, 1],
        "PC3": embeddings_3d[:, 2],
        "Plot Status": label_names,
    })

    fig_3d = px.scatter_3d(
        df_pca_3d, x="PC1", y="PC2", z="PC3",
        color="Plot Status",
        title=f"3D PCA Projection (Test Set)<br><sup>Variance: PC1={var_ratio_3d[0]:.2f}%, PC2={var_ratio_3d[1]:.2f}%, PC3={var_ratio_3d[2]:.2f}%</sup>",
        opacity=0.8,
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig_3d.update_traces(marker=dict(size=4))
    fig_3d.update_layout(margin=dict(l=0, r=0, t=40, b=0))

    pca_3d_path = os.path.join(output_dir, "pca_embeddings_3d_test.html")
    fig_3d.write_html(str(pca_3d_path), include_plotlyjs="cdn")
    print(f"PCA plots saved to: {output_dir}")


def run_defoliated_classification():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    full_dataset = DefoliatedPlotDataset(
        root_dir=DATASET_DIR,
        json_path=JSON_ANNOTATION_PATH,
        label_map=LABEL_MAPPING,
        image_size=IMAGE_SIZE,
    )

    # ---------------------------------------------------------
    # Strict 50/50 Stratified Split
    # ---------------------------------------------------------
    targets = [full_dataset.samples[i][1] for i in range(len(full_dataset))]
    
    # Train_test_split ensures exact 50% split and maintains class distribution (stratify)
    train_idx, test_idx = train_test_split(
        np.arange(len(targets)),
        test_size=TEST_SPLIT_RATIO,
        stratify=targets,
        random_state=RANDOM_SEED
    )

    train_sub = Subset(full_dataset, train_idx)
    test_sub = Subset(full_dataset, test_idx)

    print(f"\nDataset strictly split: {len(train_sub)} Training samples | {len(test_sub)} Testing samples")

    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_sub, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    # Initialize model
    model = LinearPlotClassifier(
        checkpoint_path=CHECKPOINT_PATH,
        num_classes=NUM_CLASSES,
        dropout_rate=DROPOUT_RATE,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.classifier.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    
    best_test_loss = float('inf')
    best_model_path = os.path.join(OUTPUT_DIR, "best_linear_classifier.pth")
    
    train_loss_history = []
    test_loss_history = []

    print("\nTraining Linear Probe Classifier (Evaluated on 50% Test Set)...")
    print("="*50)

    # Training Loop
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            logits = model(imgs)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_epoch_loss = running_loss / len(train_loader)
        
        # Evaluate on the 50% Test Set to track checkpointing
        _, _, test_epoch_loss = evaluate_model(model, test_loader, DEVICE)
        
        train_loss_history.append(train_epoch_loss)
        test_loss_history.append(test_epoch_loss)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] Train Loss: {train_epoch_loss:.4f} | Test Loss: {test_epoch_loss:.4f}")
        
        # Checkpoint the best model
        if test_epoch_loss < best_test_loss:
            best_test_loss = test_epoch_loss
            torch.save(model.state_dict(), best_model_path)
            print(f"   -> Checkpoint saved! New best test loss: {best_test_loss:.4f}")

    print("="*50)
    print(f"Training Complete. Best Test Loss achieved: {best_test_loss:.4f}")

    # Plot Loss Curve
    plt.figure(figsize=(8, 6))
    plt.plot(range(1, EPOCHS + 1), train_loss_history, marker='o', linestyle='-', color='b', label='Train Loss')
    plt.plot(range(1, EPOCHS + 1), test_loss_history, marker='s', linestyle='--', color='r', label='Test Loss')
    plt.title("Loss Curve (50/50 Split)")
    plt.xlabel("Epoch")
    plt.ylabel("Cross Entropy Loss")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    
    loss_plot_path = os.path.join(OUTPUT_DIR, "loss_curve.png")
    plt.savefig(loss_plot_path, dpi=300)
    plt.close()

    # Load the best model weights before computing final metrics
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    print("\nLoaded best model weights for final evaluation.")
    
    # Generate the classification report and confusion matrix on the TEST set
    test_preds, test_labels, _ = evaluate_model(model, test_loader, DEVICE)
    generate_classification_metrics(
        all_labels=test_labels, 
        all_preds=test_preds, 
        class_names=CLASS_NAMES, 
        output_dir=OUTPUT_DIR
    )

    # Generate the PCA plots using ONLY the TEST set to avoid data leakage claims
    plot_pca_embeddings_2d_and_3d(
        model=model, 
        dataloader=test_loader, 
        device=DEVICE, 
        class_names=CLASS_NAMES, 
        output_dir=OUTPUT_DIR
    )


if __name__ == "__main__":
    run_defoliated_classification()

Total Dataset loaded: 177 valid samples.

Dataset strictly split: 88 Training samples | 89 Testing samples

Training Linear Probe Classifier (Evaluated on 50% Test Set)...
Epoch [01/50] Train Loss: 1.2995 | Test Loss: 1.1087
   -> Checkpoint saved! New best test loss: 1.1087
Epoch [02/50] Train Loss: 0.8928 | Test Loss: 1.1743
Epoch [03/50] Train Loss: 0.8924 | Test Loss: 1.2488
Epoch [04/50] Train Loss: 0.8055 | Test Loss: 1.2684
Epoch [05/50] Train Loss: 0.7679 | Test Loss: 1.2830
Epoch [06/50] Train Loss: 0.6995 | Test Loss: 1.1394
Epoch [07/50] Train Loss: 0.7018 | Test Loss: 1.0041
   -> Checkpoint saved! New best test loss: 1.0041
Epoch [08/50] Train Loss: 0.6768 | Test Loss: 0.9664
   -> Checkpoint saved! New best test loss: 0.9664
Epoch [09/50] Train Loss: 0.5570 | Test Loss: 1.2054
Epoch [10/50] Train Loss: 0.5876 | Test Loss: 0.9469
   -> Checkpoint saved! New best test loss: 0.9469
Epoch [11/50] Train Loss: 0.5469 | Test Loss: 0.8165
   -> Checkpoint saved! New best test los

Extracting Test Embeddings for PCA: 100%|██████████| 6/6 [00:02<00:00,  2.17it/s]


PCA plots saved to: ./classification_results_50_50
